<a href="https://colab.research.google.com/github/32230148/UAS_MCL/blob/main/UAS_MCL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kagglehub -q
!pip install Sastrawi -q
!pip install scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 3.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import re
import string
import os
import kagglehub
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score

from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

In [ ]:
path = kagglehub.dataset_download("satyaahb/ecommerce-ratings-and-reviews-in-bahasa-indonesia")
print("Dataset path:", path)

files = os.listdir(path)
print("Files:", files)

df = pd.read_csv(os.path.join(path, 'raw_review_googleplay.csv'))
df.head()

100%|██████████| 74.7M/74.7M [00:00<00:00, 92.7MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/satyaahb/ecommerce-ratings-and-reviews-in-bahasa-indonesia/versions/1
Files: ['raw_review_googleplay.csv']


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,f1205de2-9298-470e-8da8-5eb43e302c02,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,bakalan langganan belanja disini,5,0,NaN,2024-09-18 10:52:28,"Hi Toppers, terima kasih untuk rating dan ulas...",2024-09-18 11:20:01,NaN
1,0588c5ca-887c-4594-92e5-a39e72721396,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Saya dengar ada promo pengguna baru diskon sam...,3,0,3.280.0,2024-09-18 10:48:29,"Makasih ratingnya, Toppers. Ke depannya kami a...",2024-09-18 11:20:02,3.280.0
2,ce309431-c05d-4e53-8b27-8b159ae5a3fa,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,mantap,5,0,3.280.0,2024-09-18 10:41:53,"Hi Toppers, terima kasih untuk rating dan ulas...",2024-09-18 11:00:05,3.280.0
3,1bcb43b6-09b1-481e-89d1-b04bdf69f400,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,banyak diskon,5,0,NaN,2024-09-18 10:41:08,Terima kasih sudah mempercayakan Tokopedia seb...,2024-09-18 11:00:10,NaN
4,2414fbdb-509d-4b79-8d73-3c4cc3de7eb3,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Kurang Transparan TAGIHAN TOKOPEDIA CARD, apa ...",2,0,NaN,2024-09-18 10:41:01,"Hi Toppers, silakan sampaikan kritik/saran ata...",2024-09-18 11:00:11,NaN


In [ ]:
df = df[['content', 'score']]

df = df[df['score'] != 3].reset_index(drop=True)

df['sentiment'] = df['score'].apply(lambda x: 'positif' if x >=4 else 'negatif')

df['sentiment'].value_counts()

,count
sentiment,
positif,992077
negatif,161076


In [ ]:
from sklearn.utils import resample

df_pos = df[df['sentiment'] == 'positif']
df_neg = df[df['sentiment'] == 'negatif']

df_pos_down = resample(df_pos,
                       replace=False,
                       n_samples=len(df_neg),
                       random_state=42)


df_balanced = pd.concat([df_pos_down, df_neg]).sample(frac=1, random_state=42)

print(df_balanced['sentiment'].value_counts())

sentiment
positif    161076
negatif    161076
Name: count, dtype: int64


In [ ]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

stop_factory = StopWordRemoverFactory()
stopwords = stop_factory.get_stop_words()

In [ ]:
class TextPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, stopwords=None, stemmer=None):
        self.stopwords = stopwords
        self.stemmer = stemmer

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.apply(self._preprocess)

    def _preprocess(self, text):
        text = str(text).lower()

        emoji_pattern = re.compile(
            "["
            "\U0001F600-\U0001F64F"
            "\U0001F300-\U0001F5FF"
            "\U0001F680-\U0001F6FF"
            "\U0001F1E0-\U0001F1FF"
            "]+", flags=re.UNICODE)
        text = emoji_pattern.sub(r'', text)

        text = re.sub(r'\d+', '', text)

        text = text.translate(str.maketrans('', '', string.punctuation))

        text = re.sub(r'(.)\1{2,}', r'\1', text)

        text = text.strip()

        tokens = text.split()

        if self.stopwords:
            tokens = [w for w in tokens if w not in self.stopwords]

        if self.stemmer:
            tokens = [self.stemmer.stem(w) for w in tokens]

        return ' '.join(tokens)

In [ ]:
X = df['content']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train class distribusi:\n", y_train.value_counts())
print("Test class distribusi:\n", y_test.value_counts())

Train class distribusi:
 sentiment
positif    793661
negatif    128861
Name: count, dtype: int64
Test class distribusi:
 sentiment
positif    198416
negatif     32215
Name: count, dtype: int64


In [ ]:
pipeline = Pipeline([
    ('preprocess', TextPreprocessor(stopwords=stopwords, stemmer=stemmer)),
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=5000)),
    ('nb', MultinomialNB())
])

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

NotFittedError: Pipeline is not fitted yet.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')

print("Cross Validation Accuracy per fold:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())

Cross Validation Accuracy per fold: [0.93624014 0.93595397 0.93612307 0.93575857 0.93655639]
Mean CV Accuracy: 0.9361264290706481


In [ ]:
review_baru = [
    "produk bagus banget dan pengiriman cepat",
    "barang rusak kualitas jelek"
]

prediksi = pipeline.predict(pd.Series(review_baru))

for r, p in zip(review_baru, prediksi):
    print("Review:", r)
    print("Sentimen:", p)

Review: produk bagus banget dan pengiriman cepat
Sentimen: positif
Review: barang rusak kualitas jelek
Sentimen: negatif


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

save_folder = '/content/drive/MyDrive/MODEL_UAS_MCL/'
os.makedirs(save_folder, exist_ok=True)

joblib.dump(pipeline, os.path.join(save_folder, 'naive_bayes_pipeline.pkl'))
print("Pipeline berhasil disimpan di Drive!")

Mounted at /content/drive


In [ ]:
pipeline_path = os.path.join(save_folder, 'naive_bayes_pipeline.pkl')
pipeline_loaded = joblib.load(pipeline_path)
print("Pipeline berhasil dimuat dari Drive!")

Pipeline berhasil dimuat dari Drive!


In [ ]:
review_baru = [
    "produk bagus banget dan pengiriman cepat",
    "barang rusak kualitas sampah"
]

prediksi = pipeline_loaded.predict(pd.Series(review_baru))

for r, p in zip(review_baru, prediksi):
    print("Review:", r)
    print("Sentimen:", p)

Review: produk bagus banget dan pengiriman cepat
Sentimen: positif
Review: barang rusak kualitas sampah
Sentimen: negatif
